# Neo4j MCP Agent with Text2Cypher

This notebook connects a Strands Agent to a Neo4j knowledge graph through the **Model Context Protocol (MCP)**, then builds an autonomous **Text2Cypher** agent that discovers the graph schema and writes its own Cypher queries.

Earlier labs used the neo4j-graphrag library with a direct Python driver, calling retriever classes that handled embedding, vector search, and graph traversal internally. Here you connect to the same graph through MCP, an open standard that defines how AI agents discover and interact with external tools and data sources.

**Learning Objectives:**
- Connect a Strands Agent to a Neo4j MCP Server via Streamable HTTP transport
- Discover tools automatically with `list_tools_sync()`: `get-schema` and `read-cypher`
- Understand MCP's three-component architecture: Agent, MCP Server, Data Source
- Build a Text2Cypher agent that inspects the schema and writes original Cypher from scratch

**How it works:**
1. `MCPClient` connects to the Neo4j MCP Server through an AWS AgentCore Gateway
2. `list_tools_sync()` discovers the available tools
3. The discovered tools are passed directly to the Strands `Agent`, with no `@tool` wrappers needed
4. The agent follows a schema-first approach: it retrieves the schema, reasons about your question, then writes and runs Cypher

Strands takes a model-driven approach: the model decides what to do at each step rather than following a developer-defined workflow. Its native MCP support means the agent discovers Neo4j's tools at runtime and calls them directly.

> **Note:** LangGraph is a viable alternative framework for this pattern. It provides fine-grained control over the agent loop and is better suited for complex, multi-step workflows. This workshop uses Strands for its simpler API and built-in MCP support.

**Prerequisites:**
- `../CONFIG.txt` configured with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

> The database and indexes are already set up via the pre-deployed MCP server. You do not need to load data yourself.

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx -q

## 1. Configuration and Imports

In [ ]:
import os

from dotenv import load_dotenv
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

assert MCP_GATEWAY_URL, "MCP_GATEWAY_URL not configured in CONFIG.txt"
assert MCP_ACCESS_TOKEN, "MCP_ACCESS_TOKEN not configured in CONFIG.txt"

print(f"Model:     {MODEL_ID}")
print(f"Region:    {REGION}")
print("\nConfiguration loaded!")

## 2. Initialize Model and MCP Connection

- **BedrockModel**: Configures Claude on Amazon Bedrock with temperature 0 for deterministic responses.
- **MCPClient**: Connects to the Neo4j MCP Server and discovers available tools. `list_tools_sync()` returns tool wrappers that the agent can call directly. This is the standard Strands pattern for MCP integration.

The server exposes two tools: `get-schema` (reads the graph structure) and `read-cypher` (runs a read-only Cypher query). The agent discovers them automatically through the MCP protocol, so it does not need to know about them in advance.

In [ ]:
from botocore.config import Config as BotocoreConfig

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
    boto_client_config=BotocoreConfig(read_timeout=300),
)

# Open a persistent MCP connection for the notebook session
mcp_client = MCPClient(lambda: streamablehttp_client(
    url=MCP_GATEWAY_URL,
    headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
))
mcp_client.__enter__()

# Discover available tools from the MCP server
mcp_tools = mcp_client.list_tools_sync()
tool_names = [t.tool_name for t in mcp_tools]
print(f"MCP tools discovered: {tool_names}")
print(f"\nModel: {MODEL_ID}")

## 3. Create the Text2Cypher Agent

Pass the discovered MCP tools directly to the agent. The agent inspects each tool's name, description, and parameter schema to decide when and how to call it, with no `@tool` wrappers or manual tool selection needed.

The system prompt enforces a **schema-first approach**: the agent always retrieves the database schema before writing any Cypher. This grounds queries in the actual graph structure, so the agent uses correct node labels, relationship types, and property names rather than guessing. This is the **Text2Cypher** pattern: the agent writes original Cypher from scratch instead of selecting from pre-written templates.

In [ ]:
SYSTEM_PROMPT = """You are a Neo4j database assistant with access to a knowledge graph
containing SEC 10-K financial filing data (companies, products, services, risk factors,
financial metrics, executives, asset manager holdings).

Rules:
1. Always retrieve the database schema before writing Cypher queries.
2. Only use read-only Cypher (MATCH, RETURN, WITH, WHERE, ORDER BY, LIMIT).
3. Include LIMIT clauses to avoid excessive results.
4. Use COALESCE() or IS NOT NULL for properties that might be missing.
5. Format results clearly and cite actual data from query results.
6. Modern Cypher syntax:
   - Use elementId(n) instead of id(n); id() is removed in Neo4j 5+
   - Use COUNT{ pattern } instead of size((pattern)) for counting pattern occurrences
   - Use EXISTS{ pattern } instead of exists((pattern)) for checking pattern existence
   - Always use $parameter syntax for dynamic values, never string concatenation
"""

agent = Agent(
    model=bedrock_model,
    system_prompt=SYSTEM_PROMPT,
    tools=mcp_tools,
)


def query(question: str):
    """Ask the agent a question about the knowledge graph."""
    print(f'Question: "{question}"')
    print("-" * 60)
    response = agent(question)
    print(f"\n{response}")
    return response


print("Agent ready!")

## 4. Query the Knowledge Graph

Run the following queries to watch the agent discover the schema and write its own Cypher. Watch the tool calls in the output: the agent calls `get-schema` to learn the graph structure, then `read-cypher` to run the query it generates.

In [ ]:
# Schema discovery: the agent calls get-schema and summarizes the graph structure
query("What is the database schema? Give me a brief summary of the node labels, relationship types, and key properties.")

In [ ]:
# Simple query: the agent writes and executes Cypher via read-cypher
query("How many companies are in the database?")

In [ ]:
query("How many nodes are there by label?")

## 5. Your Query

Try your own questions about the SEC financial data:

- "What products does Apple offer?"
- "Which asset managers own stakes in NVIDIA?"
- "What risk factors does Microsoft face?"
- "Show me the financial metrics for Tesla."
- "Who are the executives at Amazon?"
- "Which companies face risk factors related to cybersecurity?"

In [ ]:
# query("Your question here")

## Summary

You connected a Strands Agent to a Neo4j MCP Server and built an autonomous Text2Cypher agent:

| Component | Role |
|-----------|------|
| **Strands Agent** | Receives your question, retrieves the schema, writes Cypher, executes it, interprets results |
| **MCPClient** | Discovers tools from the MCP server and provides them to the agent |
| **Neo4j MCP Server** | Exposes `get-schema` and `read-cypher` tools over MCP |
| **BedrockModel** | Powers the agent's reasoning with Claude on Amazon Bedrock |

The schema-first approach is critical for Text2Cypher: by retrieving the graph structure before writing queries, the agent uses correct node labels, relationship types, and property names rather than guessing. Without it, an agent might generate `MATCH (c:Corp)-[:HAS_PRODUCT]->(p:Product)` when the real label is `Company` and the relationship is `OFFERS`. The query runs without error but returns zero results.

This completes Lab 6. You progressed from MCP tool discovery to fully autonomous query generation: the Text2Cypher retrieval pattern delivered over the Model Context Protocol.

In [ ]:
# mcp_client.__exit__(None, None, None)